# 13 - A sample for reading, and facts that may help

Notebook 12 found that the evidence the reader is shown says little about the 203 accounts of the experiment, which were chosen by the ML model's mistakes. To ask whether a reader can read evidence at all, this notebook draws a new sample by what an account is: every laundering pattern type, positives outside the pattern file, and clean accounts. It then computes facts that the prompt table does not show and measures, with the classifier of notebook 12, whether they raise the ceiling.

Two disjoint samples are drawn: a development sample, on which the facts and later the guide are chosen, and a test sample, which is read only once with the fixed prompt. Only the development sample is looked at here. No model score chooses an account. Exploratory, written after notebook 11.

In [1]:
%%time
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from skeptical_second_reader.data import (
    ACCOUNTS_CSV,
    PATTERNS_TXT,
    account_entities,
    account_labels,
    load_transactions,
    pattern_members,
    read_accounts,
    read_patterns,
)
from skeptical_second_reader.facts import FACTS, account_facts
from skeptical_second_reader.model import fit_xgboost, precision_threshold
from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.reading_sample import (
    MATCHED_NAME,
    OUTSIDE_NAME,
    RANDOM_NAME,
    account_patterns,
    reading_samples,
)
from skeptical_second_reader.split import (
    N_FOLDS,
    cached_folds,
    cv_predict,
    split_groups,
)
from skeptical_second_reader.transactions import account_positions, transaction_table

df = load_transactions()
labels = account_labels(df)
owners = account_entities(labels.index, read_accounts(ACCOUNTS_CSV))
patterns = read_patterns(PATTERNS_TXT)
members = pattern_members(patterns)
groups = split_groups(owners, members)
reading = pd.read_parquet(DATA_INTERIM / "transaction_reading.parquet")
accounts = reading.index
y = labels.loc[accounts]
fold_of = cached_folds(labels, groups, DATA_INTERIM).loc[accounts]
table = transaction_table(df)
del df
sender, receiver = account_positions(table, accounts)
X = pd.read_parquet(DATA_INTERIM / "ceiling_features.parquet")
assert X.index.equals(accounts)
in_file = pd.Series(accounts.isin(members["account"]), index=accounts)
kind_of = account_patterns(patterns, members).reindex(accounts).dropna()
print(f"accounts {len(y):,}  positives {y.sum():,}")
print("accounts in the file with one pattern type:", len(kind_of))

accounts 515,078  positives 6,347
accounts in the file with one pattern type: 3068
CPU times: total: 28.7 s
Wall time: 26.8 s


## 1. The two samples

15 positives of each pattern type, 15 positives outside the pattern file, 60 clean accounts with the activity of chosen positives (number of transactions in the window) and 60 clean accounts at random. An account in attempts of two or more pattern types has no single type and is not drawn for a type. Clean accounts are outside the pattern file. The accounts of the model check and of the experiment of notebook 10 are left out, so the samples are new to the reader.

In [2]:
activity = X["transactions"].astype(int)
used = pd.read_parquet(DATA_INTERIM / "llm_experiment_cases.parquet").index.union(
    pd.read_parquet(DATA_INTERIM / "llm_trial_cases.parquet").index
)
dev, test = reading_samples(
    y, activity, kind_of, in_file, dev_seed=11, test_seed=12, exclude=used
)
assert not set(dev.index) & set(test.index)
assert not (set(dev.index) | set(test.index)) & set(used)
clean_names = [MATCHED_NAME, RANDOM_NAME]
for sample in (dev, test):
    assert (y[sample[~sample.isin(clean_names)].index] == 1).all()
    assert (y[sample[sample.isin(clean_names)].index] == 0).all()
    assert not in_file[sample[sample == OUTSIDE_NAME].index].any()
dev.to_frame().to_parquet(DATA_INTERIM / "reading_dev_cases.parquet")
test.to_frame().to_parquet(DATA_INTERIM / "reading_test_cases.parquet")
sizes = pd.DataFrame(
    {
        "development": dev.value_counts(),
        "test": test.value_counts(),
        "median transactions, development": activity[dev.index].groupby(dev).median(),
    }
)
print(sizes.to_string())
print("accounts:", len(dev), len(test))

                            development  test  median transactions, development
stratum                                                                        
clean, matched activity              60    60                              21.5
clean, random                        60    60                              16.0
pattern BIPARTITE                    15    15                              23.0
pattern CYCLE                        15    15                              41.0
pattern FAN-IN                       15    15                              21.0
pattern FAN-OUT                      15    15                               6.0
pattern GATHER-SCATTER               15    15                              22.0
pattern RANDOM                       15    15                              32.0
pattern SCATTER-GATHER               15    15                              27.0
pattern STACK                        15    15                              26.0
positive, outside the file           15 

## 2. The facts

Computed by `facts.account_facts` from all transactions of the window, without labels. Medians per stratum of the development sample only; the test sample is not looked at. These are the numbers a guide would point the reader to.

In [3]:
%%time
facts = account_facts(table, sender, receiver, accounts)
facts.to_parquet(DATA_INTERIM / "account_facts.parquet")
print(facts.loc[dev.index].groupby(dev).median().T.round(2).to_string())

stratum                   clean, matched activity  clean, random  pattern BIPARTITE  pattern CYCLE  pattern FAN-IN  pattern FAN-OUT  pattern GATHER-SCATTER  pattern RANDOM  pattern SCATTER-GATHER  pattern STACK  positive, outside the file
payers                                       2.00           1.00                2.0           3.00            2.00              2.0                    2.00            3.00                    2.00           2.00                        2.00
payees                                       1.00           1.00                1.0           2.00            2.00              1.0                    1.00            2.00                    2.00           1.00                        1.00
both_ways                                    0.00           0.00                0.0           0.00            0.00              0.0                    0.00            0.00                    0.00           0.00                        0.00
share_sent_on_within_day                    

## 3. Do the facts raise the ceiling?

The classifier of notebook 12 with and without the facts, on the same folds and threshold rule. On all accounts: test AUC-PR per round, and the recall of each pattern type and of positives outside the file at the threshold. On the development sample: the share flagged per stratum, which is the recall for positive strata and the false alarm rate for clean ones, and AUC-ROC. The test sample makes up 0.05% of all accounts and stays in the folds; its results are not shown.

In [4]:
%%time
with_facts = pd.concat([X, facts], axis=1)
variants = {
    "prompt evidence": list(X.columns),
    "evidence and facts": list(X.columns) + FACTS,
}
ceilings = {}
for name, columns in variants.items():
    cv = cv_predict(with_facts[columns], y, fold_of, fit_xgboost, precision_threshold)
    ceilings[name] = cv
    aucpr = [
        average_precision_score(y[fold_of == k], cv.scores[fold_of == k])
        for k in range(N_FOLDS)
    ]
    print(f"{name}: AUC-PR {np.mean(aucpr):.4f} +- {np.std(aucpr):.4f}")

positive_kind = kind_of[y[kind_of.index] == 1]
outside = y[(y == 1) & ~in_file].index
recall = {}
for name, cv in ceilings.items():
    flagged = cv.flags == 1
    by_kind = flagged[positive_kind.index].groupby(positive_kind).mean()
    by_kind["outside the file"] = flagged[outside].mean()
    recall[name] = by_kind
recall = pd.DataFrame(recall)
recall["positives"] = positive_kind.value_counts().reindex(recall.index)
recall.loc["outside the file", "positives"] = len(outside)
print()
print("recall at the threshold, all accounts")
print(recall.round(3).to_string())

prompt evidence: AUC-PR 0.3026 +- 0.0242


evidence and facts: AUC-PR 0.3288 +- 0.0200

recall at the threshold, all accounts
                  prompt evidence  evidence and facts  positives
pattern                                                         
BIPARTITE                   0.324               0.327      441.0
CYCLE                       0.603               0.608      232.0
FAN-IN                      0.527               0.523      298.0
FAN-OUT                     0.146               0.279      315.0
GATHER-SCATTER              0.326               0.367      641.0
RANDOM                      0.467               0.461      180.0
SCATTER-GATHER              0.559               0.604      338.0
STACK                       0.448               0.445      623.0
outside the file            0.011               0.015     3187.0
CPU times: total: 18min 31s
Wall time: 1min 20s


In [5]:
truth = y[dev.index]
flagged = pd.DataFrame(
    {name: cv.flags[dev.index] == 1 for name, cv in ceilings.items()}
).groupby(dev)
print("share flagged per stratum, development sample")
print(flagged.mean().round(3).to_string())
print()
print("AUC-ROC on the development sample")
for name, cv in ceilings.items():
    print(f"  {name}: {roc_auc_score(truth, cv.scores[dev.index]):.3f}")

share flagged per stratum, development sample
                            prompt evidence  evidence and facts
stratum                                                        
clean, matched activity               0.017               0.000
clean, random                         0.000               0.000
pattern BIPARTITE                     0.267               0.267
pattern CYCLE                         0.600               0.533
pattern FAN-IN                        0.800               0.800
pattern FAN-OUT                       0.200               0.400
pattern GATHER-SCATTER                0.133               0.200
pattern RANDOM                        0.333               0.333
pattern SCATTER-GATHER                0.667               0.667
pattern STACK                         0.267               0.333
positive, outside the file            0.067               0.067

AUC-ROC on the development sample
  prompt evidence: 0.857
  evidence and facts: 0.876
